**Name:** Haider Ali  
**Roll No:** 23-NTU-CS-FL-1254  
**Lab:** PCA and Random Forest

## Question 1: PCA on Fashion MNIST

In [ ]:
import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, ConfusionMatrixDisplay,
                             mean_absolute_error, mean_squared_error, r2_score)

def find_file(names):
    folders = [".", "/content", "/kaggle/input", "/mnt/user-data/uploads", os.path.expanduser("~/Downloads")]
    for folder in folders:
        for root, _, files in os.walk(folder):
            for n in names:
                if n in files:
                    return os.path.join(root, n)
    return None

In [ ]:
path = find_file(["fashion-mnist_train.csv", "fashion_mnist_train.csv", "fashion-mnist.csv"])
if path:
    fm = pd.read_csv(path)
else:
    from sklearn.datasets import fetch_openml
    raw = fetch_openml("Fashion-MNIST", version=1, as_frame=True)
    fm = raw.data.copy()
    fm.insert(0, "label", raw.target.astype(int).values)

print("Shape:", fm.shape)
print(fm.head(10))
print()
print("Class distribution:")
print(fm["label"].value_counts().sort_index())

In [ ]:
class_names = ["T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
               "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"]

y_fm = fm["label"].values
X_fm = fm.drop(columns=["label"]).values.astype("float32") / 255.0

scaler = StandardScaler()
X_fm_scaled = scaler.fit_transform(X_fm)

print("Features shape:", X_fm.shape)
print("Pixel range after normalization:", X_fm.min(), "to", X_fm.max())
print("Mean after scaling:", round(float(X_fm_scaled.mean()), 4))
print("Std after scaling:", round(float(X_fm_scaled.std()), 4))

Scaling puts every pixel on the same scale with mean 0 and standard deviation 1. PCA picks directions of maximum variance, so without scaling the pixels with larger spread dominate the components. After scaling every pixel gets an equal say, but pixels that are almost always zero (borders) get stretched too, which can add some noise.

In [ ]:
n_features = X_fm_scaled.shape[1]

pca_fixed = PCA(n_components=50)
Z_fixed = pca_fixed.fit_transform(X_fm_scaled)

pca_var = PCA(n_components=0.95)
Z_var = pca_var.fit_transform(X_fm_scaled)

pca_all = PCA(n_components=None)
Z_all = pca_all.fit_transform(X_fm_scaled)

results = []
for name, p in [("Method 1: n_components=50", pca_fixed),
                ("Method 2: n_components=0.95", pca_var),
                ("Method 3: n_components=None", pca_all)]:
    k = p.n_components_
    cum = float(np.sum(p.explained_variance_ratio_))
    print(name)
    print("  Original features:", n_features)
    print("  Retained components:", k)
    print("  Cumulative explained variance:", round(cum, 4))
    results.append({"Method": name,
                    "Original Features": n_features,
                    "Retained Components": k,
                    "Cumulative Variance": round(cum, 4),
                    "Reduction (%)": round((1 - k / n_features) * 100, 2)})

In [ ]:
cum_all = np.cumsum(pca_all.explained_variance_ratio_)

plt.figure(figsize=(9, 5))
plt.plot(range(1, len(cum_all) + 1), cum_all)
plt.axhline(0.95, color="red", linestyle="--", label="95% variance")
plt.axvline(pca_var.n_components_, color="green", linestyle="--",
            label=str(pca_var.n_components_) + " components")
plt.axvline(50, color="orange", linestyle="--", label="50 components")
plt.xlabel("Number of Principal Components")
plt.ylabel("Cumulative Explained Variance")
plt.title("Cumulative Explained Variance (Fashion MNIST)")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

In [ ]:
rng = np.random.RandomState(42)
idx = rng.choice(len(Z_all), size=min(10000, len(Z_all)), replace=False)

plt.figure(figsize=(10, 7))
sc = plt.scatter(Z_all[idx, 0], Z_all[idx, 1], c=y_fm[idx], cmap="tab10", s=6, alpha=0.7)
handles = [plt.Line2D([0], [0], marker="o", linestyle="", color=sc.cmap(sc.norm(i)), label=class_names[i])
           for i in range(10)]
plt.legend(handles=handles, title="Class", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.xlabel("First Principal Component")
plt.ylabel("Second Principal Component")
plt.title("Fashion MNIST projected on first two principal components")
plt.grid(alpha=0.2)
plt.tight_layout()
plt.show()

print("Variance in first two components:",
      round(float(np.sum(pca_all.explained_variance_ratio_[:2])) * 100, 2), "%")

In [ ]:
comparison = pd.DataFrame(results)
print(comparison.to_string(index=False))

most_reduction = comparison.loc[comparison["Reduction (%)"].idxmax(), "Method"]
most_info = comparison.loc[comparison["Cumulative Variance"].idxmax(), "Method"]
print()
print("Greatest dimensionality reduction:", most_reduction)
print("Most information preserved:", most_info)

Method 1 (50 components) gives the greatest dimensionality reduction since it keeps the fewest components, but it keeps less than 95% of the variance. Method 2 keeps at least 95% of the variance with a moderate number of components, so it is a balance between size and information. Method 3 keeps all 784 components and 100% of the variance, so it preserves the most information but does not reduce anything.

## Question 2: Random Forest Classification on Heart Disease

In [ ]:
hpath = find_file(["heart_disease_uci.csv", "heart.csv", "heart_disease.csv", "Heart_Disease.csv"])
if hpath is None:
    raise FileNotFoundError("Place the Heart Disease csv file next to this notebook")
heart = pd.read_csv(hpath)

print("Shape:", heart.shape)
print(heart.head(10))
print()
print("Missing values:")
print(heart.isnull().sum())

In [ ]:
target_col = None
for c in ["target", "num", "HeartDisease", "condition", "output"]:
    if c in heart.columns:
        target_col = c
        break
if target_col is None:
    target_col = heart.columns[-1]

heart[target_col] = (heart[target_col].astype(float) > 0).astype(int)

print("Target column:", target_col)
print(heart[target_col].value_counts())
print(heart[target_col].value_counts(normalize=True).round(3))

In [ ]:
heart_clean = heart.drop(columns=[c for c in ["id", "dataset", "Unnamed: 0"] if c in heart.columns]).copy()

for col in heart_clean.columns:
    if col == target_col:
        continue
    if pd.api.types.is_numeric_dtype(heart_clean[col]) and not pd.api.types.is_bool_dtype(heart_clean[col]):
        heart_clean[col] = heart_clean[col].fillna(heart_clean[col].median())
    else:
        heart_clean[col] = heart_clean[col].fillna(heart_clean[col].mode()[0])

heart_clean = heart_clean.drop_duplicates().reset_index(drop=True)
heart_clean = pd.get_dummies(heart_clean, drop_first=True)
heart_clean = heart_clean.astype({c: int for c in heart_clean.select_dtypes("bool").columns})

print("Shape after cleaning:", heart_clean.shape)
print("Remaining missing values:", int(heart_clean.isnull().sum().sum()))

In [ ]:
X_h = heart_clean.drop(columns=[target_col])
y_h = heart_clean[target_col]

Xh_train, Xh_test, yh_train, yh_test = train_test_split(
    X_h, y_h, test_size=0.2, stratify=y_h, random_state=42
)
print("Train:", Xh_train.shape, "Test:", Xh_test.shape)

In [ ]:
rf_basic = RandomForestClassifier(random_state=42)
rf_basic.fit(Xh_train, yh_train)
basic_acc = accuracy_score(yh_test, rf_basic.predict(Xh_test))
print("Basic Random Forest test accuracy:", round(basic_acc, 4))

In [ ]:
param_grid_clf = {
    "n_estimators": [100, 200],
    "max_depth": [None, 10, 20],
    "min_samples_split": [2, 5],
    "min_samples_leaf": [1, 2],
    "max_features": ["sqrt", "log2"]
}

grid_clf = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid_clf,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)
grid_clf.fit(Xh_train, yh_train)

print("Best hyperparameters:", grid_clf.best_params_)
print("Best cross-validation accuracy:", round(grid_clf.best_score_, 4))

In [ ]:
best_clf = grid_clf.best_estimator_
yh_pred = best_clf.predict(Xh_test)

print("Accuracy :", round(accuracy_score(yh_test, yh_pred), 4))
print("Precision:", round(precision_score(yh_test, yh_pred), 4))
print("Recall   :", round(recall_score(yh_test, yh_pred), 4))
print("F1-score :", round(f1_score(yh_test, yh_pred), 4))

cm = confusion_matrix(yh_test, yh_pred)
print("Confusion matrix:")
print(cm)

ConfusionMatrixDisplay(cm, display_labels=["No Disease", "Disease"]).plot(cmap="Blues")
plt.title("Confusion Matrix (Test Set)")
plt.show()

In [ ]:
imp = pd.Series(best_clf.feature_importances_, index=X_h.columns).sort_values(ascending=False)

plt.figure(figsize=(9, 6))
imp.head(15)[::-1].plot(kind="barh")
plt.xlabel("Importance")
plt.title("Feature Importances (Heart Disease)")
plt.tight_layout()
plt.show()

print("Three most influential features:")
print(imp.head(3))

## Question 3: Random Forest Regression on House Prices

In [ ]:
tpath = find_file(["train.csv"])
if tpath is not None:
    house = pd.read_csv(tpath)
else:
    from sklearn.datasets import fetch_openml
    house = fetch_openml("house_prices", as_frame=True).frame

print("Shape:", house.shape)
print(house.head(10))

In [ ]:
miss = house.isnull().sum()
print("Missing values:")
print(miss[miss > 0].sort_values(ascending=False))
print()
print("SalePrice summary:")
print(house["SalePrice"].describe())

In [ ]:
y_r = house["SalePrice"]
X_r = house.drop(columns=["SalePrice"])
if "Id" in X_r.columns:
    X_r = X_r.drop(columns=["Id"])

num_cols = X_r.select_dtypes(include=np.number).columns
cat_cols = X_r.select_dtypes(exclude=np.number).columns

X_r[num_cols] = X_r[num_cols].fillna(X_r[num_cols].median())
X_r[cat_cols] = X_r[cat_cols].astype(object).fillna("None")
X_r = pd.get_dummies(X_r, columns=list(cat_cols), drop_first=True)
X_r = X_r.astype({c: int for c in X_r.select_dtypes("bool").columns})

print("Features after preprocessing:", X_r.shape)
print("Remaining missing values:", int(X_r.isnull().sum().sum()))

In [ ]:
Xr_train, Xr_test, yr_train, yr_test = train_test_split(
    X_r, y_r, test_size=0.2, random_state=42
)
print("Train:", Xr_train.shape, "Test:", Xr_test.shape)

In [ ]:
def regression_scores(model, X, y):
    pred = model.predict(X)
    mse = mean_squared_error(y, pred)
    return {"MAE": mean_absolute_error(y, pred),
            "MSE": mse,
            "RMSE": np.sqrt(mse),
            "R2": r2_score(y, pred)}

rf_default = RandomForestRegressor(random_state=42)
rf_default.fit(Xr_train, yr_train)
default_scores = regression_scores(rf_default, Xr_test, yr_test)

print("Default Random Forest (test set):")
for k, v in default_scores.items():
    print(" ", k, ":", round(v, 4))

In [ ]:
param_grid_reg = {
    "n_estimators": [100, 200],
    "max_depth": [None, 10, 20],
    "min_samples_split": [2, 5],
    "min_samples_leaf": [1, 2],
    "max_features": [0.7, 1.0]
}

grid_reg = GridSearchCV(
    RandomForestRegressor(random_state=42),
    param_grid_reg,
    cv=5,
    scoring="neg_mean_squared_error",
    n_jobs=-1
)
grid_reg.fit(Xr_train, yr_train)

best_cv_mse = -grid_reg.best_score_
best_cv_rmse = np.sqrt(best_cv_mse)

print("Best hyperparameters:", grid_reg.best_params_)
print("Best CV score (neg MSE):", round(grid_reg.best_score_, 2))
print("Best CV MSE:", round(best_cv_mse, 2))
print("Best CV RMSE:", round(best_cv_rmse, 2))

In [ ]:
best_reg = grid_reg.best_estimator_
tuned_scores = regression_scores(best_reg, Xr_test, yr_test)

compare = pd.DataFrame({"Default Model": default_scores, "Optimized Model": tuned_scores}).round(4)
print(compare)

In [ ]:
pred_r = best_reg.predict(Xr_test)

plt.figure(figsize=(7, 7))
plt.scatter(yr_test, pred_r, alpha=0.6)
lims = [min(yr_test.min(), pred_r.min()), max(yr_test.max(), pred_r.max())]
plt.plot(lims, lims, "r--")
plt.xlabel("Actual SalePrice")
plt.ylabel("Predicted SalePrice")
plt.title("Actual vs Predicted House Prices")
plt.grid(alpha=0.3)
plt.show()

imp_r = pd.Series(best_reg.feature_importances_, index=X_r.columns).sort_values(ascending=False)

plt.figure(figsize=(8, 4))
imp_r.head(5)[::-1].plot(kind="barh")
plt.xlabel("Importance")
plt.title("Top 5 Feature Importances (House Prices)")
plt.tight_layout()
plt.show()

print(imp_r.head(5))